# 6 Multi-head attention

**Multi-head attention (zoom-in 1):**
$\;W_i^Q,W_i^K\in\mathbb{R}^{d_{\text{model}}\times d_{\text{k}}},\,$
$W_i^V\in\mathbb{R}^{d_{\text{model}}\times d_{\text{v}}},\,$
$W^O\in\mathbb{R}^{h d_{\text{v}}\times d_{\text{model}}}$
$$\operatorname{MultiHead}(Q,K,V)=\operatorname{Concat}(\operatorname{head}_1,\dotsc,\operatorname{head}_h)W^O\in\mathbb{R}^{n\times d_{\text{model}}}$$
$$\operatorname{head}_i=\operatorname{Attention}(QW_i^Q,KW_i^K,VW_i^V)\in\mathbb{R}^{n\times d_{\text{v}}}$$

**Multi-head attention with bias:** $\;$ Adds a bias vector to each linear transformation

**Original values:** $\;$ $d_{\text{model}}=512,\,h=8,\, d_{\text{k}}=d_{\text{v}}=\dfrac{d_{\text{model}}}{h}=64$

**Exercise:** $\;$ Compute $\,\operatorname{MultiHead}(Q,K,V)\,$ with $\,h=1,\,$ 
$Q=K=V=\begin{pmatrix}-0.7071&0.7071\\0.7071&-0.7071\\0.7070&-0.7070\end{pmatrix}\,$ and
$$\begin{align*}
W_1^Q&=\begin{pmatrix}0.8635&0.7223\\0.5531&0.3659\end{pmatrix}^t&B_1^Q&=\boldsymbol{1}_n(0.6123,-0.2899)\\
W_1^K&=\begin{pmatrix}-0.0060&-0.5075\\-0.0329&0.8903\end{pmatrix}^t&B_1^K&=\boldsymbol{1}_n(0.2253,-0.4414)\\
W_1^V&=\begin{pmatrix}0.4922&-0.3579\\-0.5233&0.0872\end{pmatrix}^t&B_1^V&=\boldsymbol{1}_n(0.0727,-0.5929)\\
W^O&=\begin{pmatrix}1.2168&-0.1905\\-0.0890&-0.5564\end{pmatrix}^t&B^O&=\boldsymbol{1}_n(-0.5157,-0.1097)
\end{align*}$$


<p style="page-break-after:always;"></p>


**Solution:**

In [1]:
import torch; import torch.nn as nn; torch.manual_seed(23)
import import_ipynb; from at251 import create_model
model = create_model(src_vocab_size=3, tgt_vocab_size=3, embed_dim=2, num_layers=1, num_heads=1, dropout=0.)
src = torch.LongTensor([[1, 2, 1]]); x = model.src_embed(src).data
norm_x = model.encoder.layers[0].norm_self_attn(x).data; query = key = value = norm_x; norm_x

tensor([[[-0.7071,  0.7071],
         [ 0.7071, -0.7071],
         [ 0.7070, -0.7070]]])

In [2]:
self_attn = model.encoder.layers[0].self_attn
bsz, seq_len, embed_dim = query.size(); num_heads = 1; head_dim = embed_dim // num_heads
print(f"bsz = {bsz}; seq_len = {seq_len}; embed_dim = {embed_dim}; num_heads = {num_heads}; head_dim = {head_dim}")

bsz = 1; seq_len = 3; embed_dim = 2; num_heads = 1; head_dim = 2


In [3]:
print("W_Q: weight", str(self_attn.q_proj.weight.data).replace('\n',''), " bias", self_attn.q_proj.bias.data)
q = self_attn.q_proj(query).view(bsz, -1, num_heads, head_dim).transpose(1, 2); q.data

W_Q: weight tensor([[0.8635, 0.7223],        [0.5531, 0.3659]])  bias tensor([ 0.6123, -0.2899])


tensor([[[[ 0.5124, -0.4223],
          [ 0.7122, -0.1575],
          [ 0.7122, -0.1575]]]])

$$\begin{align*}
QW_1^Q+B_1^Q&=\begin{pmatrix}-0.7071&0.7071\\0.7071&-0.7071\\0.7070&-0.7070\end{pmatrix}%
\begin{pmatrix}0.8635&0.7223\\0.5531&0.3659\end{pmatrix}^t+\begin{pmatrix}1\\1\\1\end{pmatrix}(0.6123,-0.2899)\\
&=\begin{pmatrix}-0.0999&-0.1324\\0.0999&0.1324\\0.0999&0.1324\end{pmatrix}%
+\begin{pmatrix}0.6123&-0.2899\\0.6123&-0.2899\\0.6123&-0.2899\end{pmatrix}%
=\begin{pmatrix}0.5124&-0.4223\\0.7122&-0.1575\\0.7122&-0.1575\end{pmatrix}
\end{align*}$$


<p style="page-break-after:always;"></p>


In [4]:
print("W_K: weight", str(self_attn.k_proj.weight.data).replace('\n',''), " bias", self_attn.k_proj.bias.data)
k = self_attn.k_proj(key).view(bsz, -1, num_heads, head_dim).transpose(1, 2); k.data

W_K: weight tensor([[-0.0060, -0.5075],        [-0.0329,  0.8903]])  bias tensor([ 0.2253, -0.4414])


tensor([[[[-0.1293,  0.2114],
          [ 0.5799, -1.0942],
          [ 0.5798, -1.0941]]]])

$$\begin{align*}
KW_1^K+B_1^K&=\begin{pmatrix}-0.7071&0.7071\\0.7071&-0.7071\\0.7070&-0.7070\end{pmatrix}%
\begin{pmatrix}-0.0060&-0.5075\\-0.0329&0.8903\end{pmatrix}^t+\begin{pmatrix}1\\1\\1\end{pmatrix}(0.2253,-0.4414)\\
&=\begin{pmatrix}-0.3546&0.6528\\0.3546&-0.6528\\0.3545&-0.6527\end{pmatrix}%
+\begin{pmatrix}0.2253&-0.4414\\0.2253&-0.4414\\0.2253&-0.4414\end{pmatrix}%
=\begin{pmatrix}-0.1293&0.2114\\0.5799&-1.0942\\0.5798&-1.0941\end{pmatrix}
\end{align*}$$

In [5]:
scores = (q @ k.transpose(-2, -1)) * head_dim**-0.5; scores.data

tensor([[[[-0.1100,  0.5368,  0.5368],
          [-0.0886,  0.4138,  0.4138],
          [-0.0886,  0.4138,  0.4138]]]])

$$\begin{align*}
&\operatorname{ScaledAttentionScores}(QW_1^Q+B_1^Q,KW_1^K+B_1^K)\\%
&\qquad=\frac{1}{\sqrt{d_{\text{k}}}}(QW_1^Q+B_1^Q)(KW_1^K+B_1^K)^t\\
&\qquad=\frac{1}{\sqrt{2}}\begin{pmatrix}0.5124&-0.4223\\0.7122&-0.1575\\0.7122&-0.1575\end{pmatrix}%
\begin{pmatrix}-0.1293&0.2114\\0.5799&-1.0942\\0.5798&-1.0941\end{pmatrix}^t%
=\begin{pmatrix}-0.1100&0.5368&0.5368\\-0.0886&0.4138&0.4138\\-0.0886&0.4138&0.4138\end{pmatrix}
\end{align*}$$


<p style="page-break-after:always;"></p>


In [6]:
attn = nn.functional.softmax(scores, dim=-1); attn.data

tensor([[[[0.2075, 0.3962, 0.3962],
          [0.2323, 0.3839, 0.3839],
          [0.2323, 0.3839, 0.3839]]]])

$$\begin{align*}
&\operatorname{AttentionWeights}(QW_1^Q+B_1^Q,KW_1^K+B_1^K)\\%
&\qquad=\operatorname{Softmax}\left(\operatorname{ScaledAttentionScores}(QW_1^Q+B_1^Q,KW_1^K+B_1^K)\right)\\
&\qquad=\operatorname{Softmax}\left(\begin{matrix}-0.1100&0.5368&0.5368\\-0.0886&0.4138&0.4138\\-0.0886&0.4138&0.4138\end{matrix}\right)%
=\begin{pmatrix}0.2075&0.3962&0.3962\\0.2323&0.3839&0.3839\\0.2323&0.3839&0.3839\end{pmatrix}
\end{align*}$$

In [7]:
print("W_V: weight", str(self_attn.v_proj.weight.data).replace('\n',''), " bias", self_attn.v_proj.bias.data)
v = self_attn.v_proj(value).view(bsz, -1, num_heads, head_dim).transpose(1, 2); v.data

W_V: weight tensor([[ 0.4922, -0.3579],        [-0.5233,  0.0872]])  bias tensor([ 0.0727, -0.5929])


tensor([[[[-0.5284, -0.1613],
          [ 0.6738, -1.0246],
          [ 0.6737, -1.0246]]]])

$$\begin{align*}
VW_1^V+B_1^V&=\begin{pmatrix}-0.7071&0.7071\\0.7071&-0.7071\\0.7070&-0.7070\end{pmatrix}%
\begin{pmatrix}0.4922&-0.3579\\-0.5233&0.0872\end{pmatrix}^t+\begin{pmatrix}1\\1\\1\end{pmatrix}(0.0727,-0.5929)\\
&=\begin{pmatrix}-0.6011&0.4317\\0.6011&-0.4317\\0.6010&-0.4316\end{pmatrix}%
+\begin{pmatrix}0.0727&-0.5929\\0.0727&-0.5929\\0.0727&-0.5929\end{pmatrix}%
=\begin{pmatrix}-0.5284&-0.1613\\0.6738&-1.0246\\0.6737&-1.0246\end{pmatrix}
\end{align*}$$


<p style="page-break-after:always;"></p>


In [8]:
values = attn @ v; values = values.transpose(1, 2).reshape(bsz, seq_len, embed_dim); values.data

tensor([[[ 0.4243, -0.8454],
         [ 0.3945, -0.8241],
         [ 0.3945, -0.8241]]])

$$\begin{align*}
\operatorname{head}_1%
&=\operatorname{Attention}(QW_1^Q+B_1^Q,KW_1^K+B_1^K,VW_1^V+B_1^V)\\
&=\operatorname{AttentionWeights}(QW_1^Q+B_1^Q,KW_1^K+B_1^K)(VW_1^V+B_1^V)\\
&=\begin{pmatrix}0.2075&0.3962&0.3962\\0.2323&0.3839&0.3839\\0.2323&0.3839&0.3839\end{pmatrix}%
\begin{pmatrix}-0.5284&-0.1613\\0.6738&-1.0246\\0.6737&-1.0246\end{pmatrix}%
=\begin{pmatrix}0.4243&-0.8454\\0.3945&-0.8241\\0.3945&-0.8241\end{pmatrix}%
\end{align*}$$

In [9]:
print("W_O: weight", str(self_attn.out_proj.weight.data).replace('\n',''), " bias", self_attn.out_proj.bias.data)
out = self_attn.out_proj(values).view(bsz, -1, num_heads, head_dim).transpose(1, 2); out.data

W_O: weight tensor([[ 1.2168, -0.1905],        [-0.0890, -0.5564]])  bias tensor([-0.5157, -0.1097])


tensor([[[[0.1616, 0.3229],
          [0.1214, 0.3137],
          [0.1214, 0.3137]]]])

$$\begin{align*}
\operatorname{MultiHead}(Q,K,V)&=\operatorname{head}_1W^O+B^O\\
&=\begin{pmatrix}0.4243&-0.8454\\0.3945&-0.8241\\0.3945&-0.8241\end{pmatrix}%
\begin{pmatrix}1.2168&-0.1905\\-0.0890&-0.5564\end{pmatrix}^t+\begin{pmatrix}1\\1\\1\end{pmatrix}(-0.5157,-0.1097)\\
&=\begin{pmatrix}0.6773&0.4327\\0.6371&0.4234\\0.6371&0.4234\end{pmatrix}%
+\begin{pmatrix}-0.5157&-0.1097\\-0.5157&-0.1097\\-0.5157&-0.1097\end{pmatrix}%
=\begin{pmatrix}0.1616&0.3229\\0.1214&0.3137\\0.1214&0.3137\end{pmatrix}
\end{align*}$$


In [10]:
str(self_attn(norm_x, norm_x, norm_x).data).replace('\n','')

'tensor([[[0.1616, 0.3229],         [0.1214, 0.3137],         [0.1214, 0.3137]]])'


<p style="page-break-after:always;"></p>
